# CS448 - Lab 9: Spectral Factorizations

## Part 1. Learning Spectral Components

In [2]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms

def stft( input_sound, dft_size, hop_size, window):
    """Compute the short-time Fourier transform of a sound.
    Args:
        input_sound: 1D numpy array containing the sound samples
        dft_size: Size of the DFT to compute (number of frequency bins)
        hop_size: Number of samples to advance between successive frames
        window: 1D numpy array containing the analysis window (length = dft_size)
    Returns:
        f: 2D numpy array (frequencies x time) containing the complex-valued STFT
    """
    input_sound_length = len(input_sound)
    num_frames = 1 + (input_sound_length - dft_size) // hop_size
    spectrogram = np.zeros((dft_size // 2 + 1, num_frames), dtype=np.complex64)
    for frame in range(num_frames):
        start = frame * hop_size
        end = start + dft_size
        segment = input_sound[start:end] * window
        spectrum = np.fft.rfft(segment, n=dft_size)
        spectrogram[:, frame] = spectrum

    # Return a complex-valued spectrogram (frequencies x time)    
    return spectrogram
    # # YOUR CODE HERE
    # raise NotImplementedError()

def istft( stft_output, dft_size, hop_size, window):
    """Compute the inverse short-time Fourier transform of a spectrogram.
    Args:
        stft_output: 2D numpy array (frequencies x time) containing the complex-valued STFT
        dft_size: Size of the DFT that was computed (number of frequency bins)
        hop_size: Number of samples advanced between successive frames
        window: 1D numpy array containing the synthesis window (length = dft_size)
    Returns:
        x: 1D numpy array containing the reconstructed sound samples
    """
    number_frames = stft_output.shape[1]
    output_length = (number_frames - 1) * hop_size + dft_size
    x = np.zeros(output_length)
    
    for frame in range(number_frames):
        start = frame * hop_size
        end = start + dft_size
        windowed_frame = np.fft.irfft(stft_output[:, frame], n=dft_size)
        x[start:end] += windowed_frame * window
    
    return x    


def sound_player( x, rate=8000, label=''):
    from IPython.display import display, Audio, HTML
    display( HTML( 
    '<style> table, th, td {border: 0px; }</style> <table><tr><td>' + label + 
    '</td><td>' + Audio( x, rate=rate)._repr_html_()[3:] + '</td></tr></table>'
    ))


In this part we will design a simple component analyzer. Use the sound file ```bach.wav```. This is the example shown in Module 10 Slide 16. We will use a spectral factorization that will allow us to extract them all. Obtain the STFT of this signal and use a DFT size of 4096, a hop size of 256 and a Hann window. This will be stored in a matrix $\mathbf F$ whose size will be $M$ by $N$. 

You now need to implement a factorization technique. This is defined as:

$$|\mathbf F | \approx \mathbf{W} \cdot \mathbf{H}$$
$$|\mathbf{F}| \in \mathbb{R}^{M\times N}_+, \mathbf{W} \in \mathbb{R}^{M\times K}_+, \mathbf{H} \in \mathbb{R}^{K\times N}_+$$

Where $\mathbb{R}^{A\times B}_+$ is the set of matrices of size $A \times B$ containing non-negative elements, and $|\mathbf{F}|$ takes the absolute value of the STFT matrix $\mathbf{F}$.  In this case we will use $K$=4 since the mix we are analyzing has four distinct sounds.  To estimate the values of $\mathbf{W}$ and $\mathbf{H}$ start by filling them with uniformly random values between 0 to 1 and iterate over the following equations:

$$\mathbf{H} = \mathbf{H} \odot \frac{\mathbf{W}^\top \cdot |\mathbf{F}|}{\mathbf{W}^\top \cdot \mathbf{W} \cdot \mathbf{H}+ \epsilon} $$
$$\mathbf{W} = \mathbf{W} \odot \frac{ |\mathbf{F}| \cdot \mathbf{H}^\top }{ \mathbf{W} \cdot \mathbf{H} \cdot \mathbf{H}^\top + \epsilon}$$

Where $\odot$ denotes element-wise multiplication, $\cdot$ means matrix multiplication, and the fraction performs element-wise division.  The constant $\epsilon$ is assigned to a small value (e.g. 1e-7) to avoid division by zero.  Iterate for about 100 times.

(Note that this training algorithm is with somewhat different update equations from the ones introduced in class. There are many different versions depending on what kind of loss function is used. This is just one of them that is using Euclidean distance.)

Plot the columns of $\mathbf{W}$ and explain what they correspond to. Plot the rows of $\mathbf{H}$ and explain them as well. You might have to run the above procedure a couple of times since in some cases the results can come up wrong. Just to be safe, run this a dozen times and show the results that are representative of the majority of the outputs (note that each time the ordering will be different, we only care about the shapes of these quantities, not their order).

You can now try to extract each component. Take each column of $\mathbf{W}$ and compute its outer product with its corresponding row of $\mathbf{H}$. This will approximate only one component of the input spectrogram. Plot all four products and explain what they look like. Use the phase of the original input to invert these resulting spectrograms to the time domain and listen to them. What do they sound like?

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()



## Part 2: Training Dictionaries for Source Separation

In this section we will design a system that separates speech of a known speaker from a known type of noise. Use ```speaker_train.wav``` to learn the dictionary that represents the speaker; Ditto for ```chimes_train.wav```. Meanwhile, ```mixture_test.wav``` contains the mixture of the two sources. This will be a mixture that we will try to separate. The rest of the data we will use for training dictionary models.

Run a factorization on each of ```speaker_train.wav``` and ```chimes_train.wav``` individually. Set your $K=40$. From these you will obtain two matrices $\mathbf{W}_s\in\mathbb{R}^{2049\times 40}_+$ and $\mathbf{W}_c\in\mathbb{R}^{2049\times 40}_+$. These are the dictionaries of the two sounds. If you visually inspect them you will see that they look a lot like representative spectra of these two sounds.

In order to resolve the mixture we need to use these dictionaries to explain its spectrogram and then only use each dictionary’s contribution to resynthesize a time signal. This essentially involves finding the $\mathbf{H}$ matrix while fixing the $\mathbf{W}$ matrix to be a concatenation of $\mathbf{W}_s$ and $\mathbf{W}_c$, i.e., $\mathbf{W}=[\mathbf{W}_s, \mathbf{W}_c]$. You can do that using the iterative approach used in the previous part, but only updating $\mathbf{H}$ and not updating $\mathbf{W}$ at every iteration. If you do this on the mixture you will ultimately get a $\mathbf{H}$ that will let us know how to combine the elements of the pretrained dictionaries to approximate the input.

To extract the two sounds you need to isolate the contribution of the two dictionaries on the mixture. That will be $|\mathbf{F}_s| = \mathbf{W}_s \cdot \mathbf{H}_s$ and $\mathbf{F}_c = \mathbf{W}_c \cdot \mathbf{H}_c$, where $\mathbf{H}_s$ corresponds to the top 40 rows of $\mathbf{H}$ and $\mathbf{H}_c$ to its bottom 40 rows. $|\mathbf{F}_s|$ and $|\mathbf{F}_c|$  will correspond to the magnitude spectrograms of the two extracted sources. Just as before use the phase of the input mixture to invert these back to the time domain and listen to them. Do they sound like they are separated? Play around with the STFT parameters until you get the best sounding results.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

